In [ ]:
import os
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np
import random
import pandas as pd

save_path = "../data/"

if not os.path.exists(save_path + "clustering/"):
        os.makedirs(save_path + "clustering/")
        
if not os.path.exists(save_path + "clustering/silhouette_score/"):
        os.makedirs(save_path + "clustering/silhouette_score/")

## Loading the data

In [ ]:
datasets = ["semcor&omsti_noun-synsets-strat", "semcor&omsti_person.n.01", 
            "cctweets-activist", "cctweets-random"]

dataset="cctweets-random"

try:
    dataset_name, filter_name = dataset.split("_")
except:
    dataset_name = dataset
    filter_name = None


df = pd.read_csv(f"{save_path}/{dataset_name}_df.csv", index_col = 0)

filter_mask = np.arange(0, len(df)) if filter_name is None else np.where(df[filter_name])


embedding_files = [file for file in os.listdir(save_path + "embedding") if dataset_name in file and file.endswith(".npy")]
embedding_files.sort()


models = ["bert-base-uncased", "microsoft-deberta-base"]

embedding_files = [file for file in embedding_files if file.split("_")[1] in models]

embedding_files

## K-Means

In [ ]:
ks = range(2, 30+1)
keep_k = [2,3,5,6,10,12,30]

for embedding_file in embedding_files:
    
    embeddings = np.load(f"{save_path}embedding/{embedding_file}")

    assert embeddings.shape[0] == len(df), "Number of vectors not equal to df length"
    
    embeddings = embeddings[filter_mask]
    
    
    silhouette_series = []
    best_silhouette = -1
    best_k = None
    best_clustering = None
    embedding_name = f"{embedding_file.split('.')[0]}_{filter_name}"
        
    for k in ks:
        clustering = KMeans(n_clusters = k, random_state = 1444).fit(embeddings)
        if len(np.unique(clustering.labels_)) != 1:
            
            silh = silhouette_score(embeddings, clustering.labels_, sample_size= 10000, random_state = 1444)
            silhouette_series.append(silh)
            if silh > best_silhouette:
                best_silhouette = silh
                best_k = k
                best_clustering = clustering
                
        else:
            silhouette_series.append(99999)
            print(f"one cluster for {embedding_name} k={k}")
        
        if k in keep_k:
            
            with open(f"{save_path}clustering/{embedding_name}_kmeans{k}-silh{str(round(silh,2))}.npy", "wb") as f:
        
                np.save(f, clustering.labels_)
            
            
    
    if best_k is not None:
        with open(f"{save_path}clustering/{embedding_name}_best-kmeans{best_k}-silh{str(round(best_silhouette,2))}.npy", "wb") as f:
        
            np.save(f, best_clustering.labels_)
        
    with open(f"{save_path}clustering/silhouette_score/{embedding_name}_silh.txt", "w") as f:
        
        [f.write(str(round(score,2)) + "\n") for score in silhouette_series]
              